In [7]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score

df = pd.read_csv('/content/student_burnout_dropout_dataset_2.csv')

df = df.drop(columns=['Student_ID', 'Dropout_Risk'])

X = df.drop(columns=['Burnout_Level'])
y = df['Burnout_Level']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=43
)

num_feat = X.select_dtypes(include='number').columns.tolist()
nom_cat_feat = X.select_dtypes(exclude='number').drop(columns=['Family_Income_Bracket']).columns.tolist()
ord_cat_feat = ['Family_Income_Bracket']

num_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('sclaer', StandardScaler())
    ]
)

nom_cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ]
)

ord_cat_pipeline = Pipeline(
    steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['Low', 'Lower-Middle', 'Middle', 'Upper-Middle', 'High']]))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_feat),
        ('nom_cat', nom_cat_pipeline, nom_cat_feat),
        ('ord_cat', ord_cat_pipeline, ord_cat_feat)
    ]
)

pipe = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression) # Initial model, will be replaced by GridSearchCV
    ]
)

param_grid = [
    {
        'model': [LogisticRegression()],
        'model__C': [0.1, 1, 10],
        'model__penalty': ['l2'],
        'model__multi_class': ['ovr', 'multinomial']
    },
    {
        'model': [DecisionTreeClassifier()],
        'model__max_depth': [3, 5, None],
        'model__min_samples_split': [2, 5, 10],
        'model__criterion': ['gini', 'entropy']
    },
    {
        'model': [RandomForestClassifier()],
        'model__n_estimators': [10, 50, 100],
        'model__max_depth': [3, 5, None],
        'model__min_samples_split': [2, 5]
    },
    {
        'model': [SVC()],
        'model__C': [0.1, 1, 10],
        'model__kernel': ['linear', 'rbf'],
        'model__gamma': ['scale', 'auto']
    }
]

grid = GridSearchCV(pipe, param_grid, scoring='f1_weighted', cv=3, n_jobs=-1) # Changed scoring to f1_weighted
model = grid.fit(X_train, y_train).best_estimator_

y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)

train_acc = accuracy_score(y_train, y_pred_train)
train_f1 = f1_score(y_train, y_pred_train, average='weighted') # Added average='weighted'

test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average='weighted') # Added average='weighted'

acc_gap = train_acc - test_acc
f1_gap = train_f1 - test_f1

print("-"*51)
print(f"{'Scenario':<15} | {'Accuracy':^15} | {'F1 Score':^15}")
print("-"*51)
print(f"{'Traning':<15} | {train_acc:^15.2f} | {train_f1:^15.2f}")
print("-"*51)
print(f"{'Test':<15} | {test_acc:^15.2f} | {test_f1:^15.2f}")
print("-"*51)
print(f"{'Gap':<15} | {acc_gap:^15.2f} | {f1_gap:^15.2f}")

# joblib.dump(model, 'model.pkl')

['Medium']
